<div style="background:#0B1F4B;padding:22px 26px;border-radius:12px;border:3px solid #C9A227;font-family:'Times New Roman',serif">
<img src="https://raw.githubusercontent.com/drssnaiml1-cell/26UAMPC204-Artificial-Intelligence/main/assets/logo.png" alt="Sanjivani University" width="92" style="float:right;background:#FFFFFF;border:2px solid #C9A227;border-radius:8px;padding:3px;margin-left:14px">
<div style="color:#C9A227;font-size:14px;letter-spacing:2px;font-weight:bold">26UAMPC204 · ARTIFICIAL INTELLIGENCE · ASSIGNMENT 3 · UNIT III · CO204.3</div>
<div style="color:#FFFFFF;font-size:30px;font-weight:bold;margin-top:6px">Logic Meets the Real World</div>
<div style="color:#F6EBC8;font-size:15px;margin-top:8px">Due: End of Week 9 · Name: ____________ · Roll No: ________</div>
<div style="clear:both"></div></div>

## 🧠 Part A — Think out of the box

Answer each in 250–400 words in the markdown cells below.

1. **[L4]** Translate the Sanjivani attendance rule (minimum attendance required to sit the exam, with medical exceptions) into first-order logic. Which parts resist precise logical encoding, and why?
2. **[L5]** LLMs often answer logic puzzles wrongly. Is a resolution theorem prover 'smarter' than an LLM? Argue both sides with a concrete example.
3. **[L6]** Design a knowledge graph of your own life (people, places, courses, skills). What new facts can you infer with three rules? What privacy risks would publishing it create?
4. **[L5]** A semantic network says 'penguins are birds' and 'birds fly'. Propose a principled way to handle exceptions without rewriting every rule, and evaluate its weaknesses.

**Answer A3.1:** _write here_

**Answer A3.2:** _write here_

**Answer A3.3:** _write here_

**Answer A3.4:** _write here_

## 💻 Part B — Open-ended coding

The starter code runs as-is and prints a baseline. Complete every `TODO`.

### B3.1 · Open-ended: your own expert knowledge base

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Encode at least 15 facts and 8 rules about a domain you know well (cricket rules, a recipe, a game). Answer five non-trivial queries with forward chaining and print the derivation trace.</div>

In [1]:
FACTS = {("course", "AI"), ("course", "DSA"), ("prereq", "DSA", "AI"), ("passed", "asha", "DSA")}
RULES = [  # (head, [body]) with variables starting with '?'
    (("can_register", "?s", "?c"), [("passed", "?s", "?p"), ("prereq", "?p", "?c")]),
]
def match(pat, fact, th):
    if len(pat) != len(fact): return None
    th = dict(th)
    for p, f in zip(pat, fact):
        p = th.get(p, p)
        if p.startswith("?"): th[p] = f
        elif p != f: return None
    return th
def forward(facts):
    facts = set(facts); changed = True
    while changed:
        changed = False
        for head, body in RULES:
            thetas = [{}]
            for lit in body:
                thetas = [t2 for t in thetas for f in facts if (t2 := match(lit, f, t)) is not None]
            for t in thetas:
                new = tuple(t.get(x, x) for x in head)
                if new not in facts: facts.add(new); changed = True; print("derived", new)
    return facts
forward(FACTS)
# TODO: replace with YOUR domain (>= 15 facts, >= 8 rules, including one recursive rule),
#       add an explanation trace for each derived fact, and pose five queries.

derived ('can_register', 'asha', 'AI')


{('can_register', 'asha', 'AI'),
 ('course', 'AI'),
 ('course', 'DSA'),
 ('passed', 'asha', 'DSA'),
 ('prereq', 'DSA', 'AI')}

In [2]:
# Your solution for B3.1


### B3.2 · Open-ended: resolution vs model checking

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Implement CNF conversion for propositional sentences and compare the speed of resolution and truth-table model checking as the number of symbols grows from 4 to 16.</div>

In [3]:
import itertools, random, time
def random_kb(n, m, seed=0):
    rnd = random.Random(seed)
    return [[(rnd.randrange(n), rnd.random() < 0.5) for _ in range(3)] for _ in range(m)]
def satisfiable_tt(kb, n):
    for vals in itertools.product([False, True], repeat=n):
        if all(any(vals[v] == sign for v, sign in cl) for cl in kb): return True
    return False
for n in (4, 8, 12, 16):
    kb = random_kb(n, int(4.2 * n)); t = time.time()
    print(f"n={n:2d}: satisfiable={satisfiable_tt(kb, n)}  truth-table time={time.time() - t:.3f}s")
# TODO 1: implement propositional resolution (or DPLL) and time it on the same KBs.
# TODO 2: plot time vs n on a log scale and explain the curve. What is special about m/n ~ 4.2?

n= 4: satisfiable=True  truth-table time=0.000s
n= 8: satisfiable=True  truth-table time=0.000s
n=12: satisfiable=False  truth-table time=0.011s
n=16: satisfiable=True  truth-table time=0.043s


In [4]:
# Your solution for B3.2


### B3.3 · Open-ended: mini ontology with SPARQL

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Build a small ontology (≥ 20 triples) for the AI&DS department and answer four questions with SPARQL queries.</div>

In [5]:
try:
    from rdflib import Graph, Namespace, RDF, Literal
    EX = Namespace("http://example.org/aids#"); g = Graph()
    g.add((EX.AI, RDF.type, EX.Course)); g.add((EX.AI, EX.hasCredits, Literal(3)))
    g.add((EX.AI, EX.taughtIn, EX.Semester3)); g.add((EX.DSA, EX.prerequisiteOf, EX.AI))
    q = "SELECT ?c WHERE { ?c <http://example.org/aids#prerequisiteOf> <http://example.org/aids#AI> }"
    print("Prerequisites of AI:", [str(r.c).split("#")[1] for r in g.query(q)])
except ImportError:
    print("Install rdflib first:  pip install rdflib")
# TODO: model courses, faculty, labs and skills (>= 20 triples) and write four SPARQL queries,
#       one of which needs a property path (e.g. prerequisites of prerequisites).

Prerequisites of AI: ['DSA']


In [6]:
# Your solution for B3.3


## 📋 Reflection

What surprised you? What would you do with one more week? How did you use AI tools, if at all?